# InstaGPT GraphRAG - Colab LLM Server

Hosts **Qwen 2.5 7B Instruct** via vLLM with an OpenAI-compatible API.

## Setup
1. Select **Runtime > Change runtime type > T4 GPU**
2. Run all cells in order
3. Copy the **public URL** from Cell 5
4. Paste it into your project's `.env` as `COLAB_BASE_URL`

In [ ]:
# Cell 1: Install dependencies with CUDA support
!nvidia-smi | head -5

# Install compatible vLLM for Colab's CUDA
!pip install torch --index-url https://download.pytorch.org/whl/cu121 -q 2>&1 | tail -3
!pip install vllm -q 2>&1 | tail -3
!pip install huggingface_hub ninja -q 2>&1 | tail -3
!apt-get install -y npm >/dev/null 2>&1

# Verify CUDA
import torch
print(f"Torch CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print('Installation complete.')

In [ ]:
# Cell 2: Download model with progress bar
from huggingface_hub import snapshot_download
import os

model_id = "Qwen/Qwen2.5-7B-Instruct"
cache_dir = "/tmp/hf_cache"

# Check if already cached
cached_path = os.path.join(cache_dir, f"models--{model_id.replace('/', '--')}")
if os.path.exists(cached_path):
    print(f"Model already cached at {cache_dir}, skipping download.")
else:
    print(f"Downloading {model_id} (~16GB)...")
    snapshot_download(model_id, cache_dir=cache_dir)
    print("Download complete!")

In [ ]:
# Cell 3: Start vLLM server (model already cached, starts fast)
import time, urllib.request

# Kill any existing vLLM process
!pkill -f vllm || true
time.sleep(2)

# Start vLLM server in background
!nohup python -m vllm.entrypoints.openai.api_server \
    --model Qwen/Qwen2.5-7B-Instruct \
    --hf-cache-dir /tmp/hf_cache \
    --host 0.0.0.0 \
    --port 8000 \
    --max-model-len 4096 \
    --gpu-memory-utilization 0.85 \
    --dtype half \
    --enforce-eager \
    > /tmp/vllm_server.log 2>&1 &

# Wait for server to be ready
print("Starting vLLM server...")
for i in range(90):  # 3 min timeout (first run loads model)
    time.sleep(2)
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=2)
        print("Server ready!")
        break
    except Exception:
        if i % 5 == 0:
            print(f"  Waiting... ({i*2}s)", end="\r")
else:
    print("Timeout. Checking logs...")
    !tail -30 /tmp/vllm_server.log

In [ ]:
# Cell 4: Verify server is running
!curl -s http://localhost:8000/v1/models | python -m json.tool

In [ ]:
# Cell 5: Create public URL via localtunnel (no auth required)
import subprocess, time, re

# Install localtunnel
!npm install -g localtunnel 2>/dev/null

# Start tunnel in background
lt_process = subprocess.Popen(
    ["lt", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

# Wait for URL
print("Creating public URL...")
public_url = None
for i in range(30):
    time.sleep(1)
    line = lt_process.stdout.readline()
    if "your url is:" in line.lower() or "https://" in line:
        match = re.search(r'https://[\w\-\.]+\.loca\.lt', line)
        if match:
            public_url = match.group(0)
            break
    # Also check stderr (localtunnel sometimes writes there)
    if lt_process.poll() is None:
        err = lt_process.stderr.readline()
        if "https://" in err:
            match = re.search(r'https://[\w\-\.]+\.loca\.lt', err)
            if match:
                public_url = match.group(0)
                break

if public_url:
    print("\n" + "="*60)
    print("COLAB SERVER IS RUNNING")
    print("="*60)
    print(f"\nPublic URL: {public_url}")
    print(f"\nAdd this to your .env file:")
    print(f"  COLAB_BASE_URL={public_url}")
    print(f"  LLM_PROVIDER=colab")
    print(f"\nKeep this tab open! Closing it stops the server.")
    print("="*60)
else:
    print("Failed to get URL. Try running this cell again.")
    print("Alternatively, use ngrok with your auth token:")
    print("  from pyngrok import ngrok")
    print("  ngrok.set_auth_token('YOUR_TOKEN')")
    print("  tunnel = ngrok.connect(8000, 'http')")
    print("  print(tunnel.public_url)")

In [ ]:
# Cell 6 (Optional): Test a chat completion
import requests, json

response = requests.post(
    "http://localhost:8000/v1/chat/completions",
    json={
        "model": "Qwen/Qwen2.5-7B-Instruct",
        "messages": [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Say hello in one sentence."}
        ],
        "temperature": 0.2,
        "max_tokens": 100
    },
    timeout=30
)

print(json.dumps(response.json(), indent=2))